# 03 · NumPy 루틴 (cupy.* · linalg · fft · random)

> **CuPy 2일 집중 코스 — Day 1 / 단원 2 (NumPy/SciPy CuPy 프로그래밍)**

CuPy 공식 [overview](https://docs.cupy.dev/en/stable/overview.html)의 **NumPy Routines** 를 본격적으로 다룹니다.
모듈 함수·선형대수·FFT·난수를 예제와 **미니앱**(PCA, 2D 디노이즈, 몬테카를로)으로 익히고, 커널 퓨전까지 맛봅니다.

### 왜 이 단원인가
`00`에서 살펴본 *CuPy 루틴 지도*(3.2절)의 두 번째 층인 "NumPy 루틴"이 바로 이 노트북의 범위입니다. `02_ndarray_core`에서
배운 배열 자체의 구조(메모리 레이아웃, dtype, 브로드캐스팅)를 바탕으로, 이제는 그 배열에 무엇을 **할 수 있는지** —
집계·선형대수·FFT·난수 — 를 실전 사례로 익힙니다. 특히 이 네 영역은 내부적으로 NVIDIA가 수십 년간 최적화해 온
검증 라이브러리(cuBLAS·cuSOLVER·cuFFT·cuRAND)를 그대로 호출하기 때문에, CuPy 코드 몇 줄만으로 고성능 구현에
바로 올라탈 수 있다는 점이 이 단원의 핵심 메시지입니다. 직접 커널을 짜는 능력(Day 2)보다 먼저, "이미 잘 만들어진
GPU 라이브러리를 최대한 활용하는 법"을 익히는 단계라고 보면 됩니다.

### 이 노트북의 흐름
모듈 함수 투어(집계/정렬/탐색/누적/집합) → 선형대수(`linalg`) + PCA 미니앱 → FFT + 2D 디노이즈 미니앱 →
난수(`random`) + 몬테카를로 미니앱 → 커널 퓨전(`@cupy.fuse`)의 순서로 진행합니다. 각 절은 "이론 배경 →
기본기 코드 → (필요하면) 미니앱 → 연습문제"의 패턴을 반복합니다 — `00`에서 소개한 학습 패턴의 연장선입니다.
미니앱 3개(PCA·2D 디노이즈·몬테카를로 π)는 모두 "실전에서 이 루틴을 어떻게 조합해 쓰는가"를 보여주기 위한
작은 응용 예제이며, 정답을 직접 채우기 전에 이론 배경 셀에서 수식과 알고리즘을 먼저 이해하는 것이 중요합니다.

## 학습 목표
- `cupy.*` 모듈 함수(집계/정렬/탐색/누적/집합)를 자유롭게 쓴다.
- `cupy.linalg`로 PCA·배치 행렬연산을, `cupy.fft`로 2D 주파수 처리를 구현한다.
- `cupy.random`으로 몬테카를로 시뮬레이션을 GPU에서 수행한다.
- `@cupy.fuse`로 원소 연산을 융합해 temporary를 줄인다.

## 목차
1. [모듈 루틴 투어 `cupy.*`](#1)
2. [선형대수 `cupy.linalg` + PCA](#2)
3. [FFT `cupy.fft` + 2D 디노이즈](#3)
4. [난수 `cupy.random` + 몬테카를로](#4)
5. [커널 퓨전 `@cupy.fuse`](#5)
6. [체크포인트](#6)

> 백엔드: cuBLAS/cuSOLVER(linalg) · cuFFT(fft) · cuRAND(random). 전체 구성은 `00`의 *CuPy 루틴 지도* 참고.

In [2]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, compare, allclose
print_env()

=== Environment ===
numpy: 2.3.5
cupy : 14.1.1
device: 0 - NVIDIA A100-SXM4-80GB (compute capability 8.0)
VRAM  : 79.25 GB


### 📦 `course_utils` 함수 — 이번 노트북에서 도입

실습 공통 유틸리티는 `course_utils.py`에 모아두고 노트북이 진행되며 **필요한 함수만 하나씩** 추가합니다. 03에서 도입하는 함수:

- **`allclose(a, b, *, rtol: float = 1e-5, atol: float = 1e-6, name: str = "")`** — CPU/GPU 결과 정확성 검증. cupy 배열은 자동으로 host로 옮겨 비교한다.

`allclose`는 `numpy.testing.assert_allclose`를 얇게 감싼 헬퍼로, 통과하면 `[allclose OK] <name>`을 출력해 어떤
검증이 성공했는지 바로 확인할 수 있게 합니다. 기본값(`rtol=1e-5, atol=1e-6`)은 단순 원소별 연산에 적합한
값이고, 이 노트북에서는 행렬곱·SVD·FFT·리덕션처럼 **연산 순서에 민감한** 함수를 검증할 때마다 호출부에서
`rtol`/`atol`을 훨씬 느슨하게(예: `2e-3`) 조정하는 것을 볼 수 있습니다 — 바로 아래 셀에서 그 이유를 설명합니다.

In [3]:
def allclose(a, b, *, rtol: float = 1e-5, atol: float = 1e-6, name: str = "") -> None:
    """CPU/GPU 결과 정확성 검증. cupy 배열은 자동으로 host로 옮겨 비교한다.

    float32 GPU 연산은 CPU와 bit 단위로 같지 않으므로 rtol/atol을 적절히 키워 사용한다.
    """
    if cp is not None:
        # a, b가 각각 cupy 배열이면 cp.asnumpy()로 device->host 복사해
        # np.testing.assert_allclose가 numpy 배열끼리 비교하도록 맞춘다.
        # (cp가 None인 환경, 즉 CuPy 미설치 시에는 cupy 배열 자체가 존재할 수 없으므로
        #  이 분기를 건너뛰고 바로 아래에서 numpy 배열끼리 비교한다.)
        if isinstance(a, cp.ndarray):
            a = cp.asnumpy(a)
        if isinstance(b, cp.ndarray):
            b = cp.asnumpy(b)
    # rtol(상대 허용오차)/atol(절대 허용오차): |a-b| <= atol + rtol*|b| 를 만족해야 통과.
    # GPU는 리덕션·행렬곱 등에서 연산 순서가 CPU와 달라 부동소수점 마지막 비트가 다를 수
    # 있으므로, 완전 일치(==)가 아니라 허용오차 이내인지를 검사한다.
    # 불일치 시 AssertionError를 발생시키며 err_msg=name으로 어떤 검증이 실패했는지 표시
    np.testing.assert_allclose(a, b, rtol=rtol, atol=atol, err_msg=name)
    # 여기까지 예외 없이 도달했다면 검증 통과 -> 성공 로그 출력
    print(f"[allclose OK] {name}".rstrip())


### (주의) CPU vs GPU 연산 결과가 미세하게 다른 이유

CPU와 GPU는 모두 **IEEE 754** 부동소수점 표준을 따릅니다. 32비트 Float을 부호(1bit) + 지수(8bit) + 가수(23bit)로 저장하는 방식도 동일합니다.

차이는 **병렬화 방식**에서 비롯됩니다.

- **CPU**: 최대 수십 개 코어로 병렬 연산
- **GPU**: 수천 개 코어로 극단적인 병렬 연산

행렬 곱셈이나 리덕션(Reduction, 총합 구하기 등) 연산에서는 **덧셈이 이루어지는 순서**가 CPU와 GPU에서 달라질 수 있습니다. 부동소수점 덧셈은 결합법칙이 항상 성립하지 않기 때문에, 순서가 달라지면 마지막 비트(LSB, Least Significant Bit)가 반올림되거나 버려지는 시점이 달라지고, 그 결과 **비트 단위의 미세한 차이**가 발생합니다.

> 즉, 하드웨어 표준(IEEE 754)이 같아도 연산 순서 차이 때문에 CPU와 GPU의 계산 결과가 완전히 동일하지 않을 수 있습니다.

### 조금 더 구체적으로 — 왜 순서가 달라지는가, 그리고 얼마나 차이 나는가

CPU에서 `numpy.sum`은 보통 배열을 앞에서부터(또는 캐시 친화적인 블록 단위로) **거의 고정된 순서**로 누적합니다.
반면 GPU 리덕션은 수천 개 스레드가 각자 일부 구간의 부분합을 계산한 뒤 **트리(tree) 형태로 절반씩 합쳐나가는**
방식을 씁니다 — 예를 들어 원소 100만 개를 더한다면, CPU는 `((((a0+a1)+a2)+a3)+...)`처럼 순차적으로,
GPU는 `(a0+a1) + (a2+a3) + ...`처럼 여러 쌍을 동시에 더한 뒤 그 결과를 다시 쌍으로 묶어 합칩니다. 수학적으로는
같은 값이어야 하지만, 부동소수점은 유효자릿수(가수 23bit ≈ 10진수 약 7자리)가 유한하기 때문에 덧셈마다
반올림 오차가 조금씩 섞여 들어가고, **더하는 순서에 따라 그 반올림이 누적되는 지점이 달라집니다**.

체감상 오차의 크기는 대략 다음과 같습니다.

| 연산 유형 | 전형적인 상대오차 | 이 노트북에서의 예 |
|---|---|---|
| 단순 원소별 연산(덧셈/곱셈 1회) | 거의 0(1 ulp 이내) | `x + 2.0` |
| 리덕션(합/평균/분산, N~10⁶) | ~1e-6 ~ 1e-5 (float32) | `cp.mean`, `cp.cumsum` |
| 행렬곱·SVD·고유분해(n~1000+) | ~1e-3 ~ 1e-2 (float32) | `solve`, `svd`, `eigh`, PCA |
| FFT(길이 N, 버터플라이 O(N log N) 단계) | ~1e-3 수준 | `fftconv`, `lowpass2d` |

행렬곱·SVD·고유분해가 리덕션보다 오차가 더 큰 이유는 내부적으로 **리덕션이 여러 단계 중첩**되기 때문입니다
(예: `n×n` 행렬곱의 각 원소는 길이 `n`짜리 내적, 즉 리덕션 한 번의 결과이고, 이런 원소가 `n²`개 모여
다시 SVD 반복계산에 입력으로 들어갑니다). 그래서 이 노트북의 `allclose` 호출을 보면 절 마다 허용오차가
다릅니다: 단순 연산은 `rtol=atol=1e-5`(00의 `feature_gpu` 연습과 동일한 기본값), 반면 `solve`/`lstsq`/
`lowpass2d`처럼 리덕션이 깊게 얽힌 연산은 `rtol=atol=2e-3` 또는 `2e-2`로 넉넉하게 잡습니다. 이 패턴은
`04_scipy_routines`의 `ndimage`/`sparse.linalg` 검증에서도 그대로 이어집니다.

> float64로 계산하면 가수부가 52bit(유효자릿수 약 15~16자리)라 오차가 훨씬 작아지지만, GPU는 float32
> 처리량이 float64보다 훨씬 높으므로(소비자용 GPU는 보통 float64가 float32 대비 1/32~1/64 속도) 이
> 코스 전체에서 float32를 기본으로 사용합니다.

<a id="1"></a>
## 1. 모듈 루틴 투어 `cupy.*`

📖 [`cupy.*` 루틴 전체](https://docs.cupy.dev/en/stable/reference/routines.html) — 대부분의 `numpy.*` 가 같은 이름으로 존재합니다.
집계·정렬·탐색·누적·집합·논리 함수를 한 번에 둘러봅니다.

### 조금 더 구체적으로 — 어떤 함수들이 있나

`cupy.*` 는 `numpy.*` 네임스페이스를 거의 그대로 미러링합니다. 이번 절에서 다루는 함수들을 성격별로 묶으면 다음과 같습니다.

| 범주 | 대표 함수 | 내부 구현 힌트 |
|---|---|---|
| 집계(reduction) | `mean`, `std`, `var`, `median`, `percentile` | 트리 리덕션(위 5절 참고) |
| 정렬 | `sort`, `argsort` | GPU 병렬 정렬(radix/merge 계열) |
| 탐색 | `argmax`, `argmin`, `searchsorted` | 정렬된 배열 이분탐색은 `searchsorted`로 O(log N) |
| 누적 | `cumsum`, `cumprod` | 병렬 prefix-sum(scan) 알고리즘 |
| 집합 | `unique` | 정렬 후 중복 제거 — 내부적으로 `sort` 1회 호출 |

`02_ndarray_core`에서 다룬 `get_array_module(x)` 패턴을 이 절 전체에서 그대로 사용합니다: 함수 시그니처가
NumPy와 동일하므로, `xp = cp.get_array_module(x)` 로 얻은 `xp`에 `np` 또는 `cp` 어느 쪽이 들어와도 같은
코드가 동작합니다(장치 비종속 코드). 아래 연습 두 개(Top-k, 이동평균)에서 바로 이 패턴을 사용합니다.

> `unique`나 `sort`처럼 **내부적으로 정렬을 수반하는 함수**는 원소 개수가 많을수록 GPU 이득이 커지지만,
> 비교연산이 분기(branch)를 유발해 단순 원소별 연산보다는 상대적으로 GPU 가속 폭이 작다는 점도 함께
> 기억해두세요 — 가속 폭이 연산 종류에 따라 왜 다른지는 `01_benchmark_basics`의 연산 강도 논의와 연결됩니다.

In [4]:
x_np = np.random.random(2_000_000).astype(np.float32)
x = cp.asarray(x_np)
print('mean/median :', float(x.mean()), float(cp.median(x)))
compare('mean', lambda: x_np.mean(), lambda: x.mean(), n_repeat=5, n_warmup=2)
compare('median', lambda: np.median(x_np), lambda: cp.median(x), n_repeat=5, n_warmup=2)

print('std/var     :', float(x.std()), float(x.var()))
compare('std', lambda: x_np.std(), lambda: x.std(), n_repeat=5, n_warmup=2)
compare('var', lambda: x_np.var(), lambda: x.var(), n_repeat=5, n_warmup=2)

print('percentile90:', float(cp.percentile(x, 90)))
compare('percentile90', lambda: np.percentile(x_np, 90), lambda: cp.percentile(x, 90), n_repeat=5, n_warmup=2)

print('argmax      :', int(cp.argmax(x)))
compare('argmax', lambda: np.argmax(x_np), lambda: cp.argmax(x), n_repeat=5, n_warmup=2)

print('cumsum[-1]  :', float(cp.cumsum(x)[-1]))
compare('cumsum', lambda: np.cumsum(x_np)[-1], lambda: cp.cumsum(x)[-1], n_repeat=5, n_warmup=2)

u = cp.unique(cp.asarray([3,1,2,3,1]))
print('unique      :', cp.asnumpy(u))
print('searchsorted:', int(cp.searchsorted(cp.sort(x), 0.5)))

mean/median : 0.5003588795661926 0.5005608797073364
std/var     : 0.2888062000274658 0.08340901881456375
percentile90: 0.9003080129623413
argmax      : 1682099
cumsum[-1]  : 1000717.8125
unique      : [1 2 3]
searchsorted: 998842


**연습 1 — Top-k**: 가장 큰 `k`개 값을 내림차순으로 반환하는 장치 비종속 함수를 완성하세요.

전체를 정렬(`sort`, O(N log N))해 뒤에서 `k`개만 취해도 되고, `k`가 `N`보다 훨씬 작다면
`argpartition`(평균 O(N))으로 "상위 k개 후보만 분리"한 뒤 그 안에서만 정렬하는 것이 이론적으로
더 효율적입니다. 다만 이 연습에서는 구현 단순성을 위해 전체 정렬 방식을 기준으로 안내합니다.
`get_array_module`을 사용해 `np`/`cp` 어디에 넘겨도 동작하는 함수로 작성하세요.

In [5]:
def topk_np(x, k): # numpy reference
    return np.sort(x)[-k:][::-1]

def topk(x, k):
    xp = cp.get_array_module(x)
    return xp.sort(x)[-k:][::-1]

x_np = np.random.randn(1_000_000).astype(np.float32)
ref = np.sort(x_np)[-5:][::-1]
out = cp.asnumpy(topk(cp.asarray(x_np), 5))
allclose(ref, out, name='topk')

    # TODO: xp.sort(x)[-k:][::-1] 또는 argpartition 사용
    # raise NotImplementedError

# x_np = np.random.randn(1_000_000).astype(np.float32)
# ref = np.sort(x_np)[-5:][::-1]; out = cp.asnumpy(topk(cp.asarray(x_np), 5))
# "CPU 결과와 GPU 결과가 똑같은지" 검증하는 코드
# allclose(ref, out, name='topk')
# compare('topk', lambda: topk_np(x_np, 5), lambda: topk(cp.asarray(x_np), 5), n_repeat=5, n_warmup=2)

[allclose OK] topk


<details><summary>💡 해답 보기</summary>

```python
def topk(x, k):
    xp = cp.get_array_module(x)
    return xp.sort(x)[-k:][::-1]
```
</details>

포인트: `xp.sort(x)`는 오름차순 정렬이므로 `[-k:]`로 가장 큰 `k`개를 뒤에서 뽑고 `[::-1]`로 뒤집어
내림차순으로 만듭니다. `get_array_module`을 함수 안에서 호출해 `x`가 `np.ndarray`든 `cp.ndarray`든
같은 코드가 그대로 동작하는 **장치 비종속(device-agnostic)** 함수가 되는 점이 핵심입니다 — 이 패턴은
`02_ndarray_core`에서 처음 소개되었고, 이 노트북 전체에서 반복해서 쓰입니다.

**연습 2 — 이동평균(cumsum 트릭)**: `cumsum`으로 길이 `w` 이동평균을 O(N)에 구하세요.

슬라이딩 윈도우 평균을 매 위치마다 `w`개씩 새로 더하면 O(N·w)이지만, **누적합(prefix sum) `c = cumsum(x)`**
를 한 번만 구해두면 구간 `[i, i+w)`의 합은 `c[i+w-1] - c[i-1]` (뺄셈 한 번)로 즉시 구할 수 있어 전체가
O(N)으로 줄어듭니다. 이는 "누적합의 차분이 구간합"이라는 성질을 이용한 전형적인 GPU 친화적 트릭입니다 —
`cumsum` 자체는 병렬 prefix-scan으로 O(log N) 단계에 계산되므로, 순차 슬라이딩 윈도우보다 훨씬 병렬화하기
좋은 형태입니다.

In [ ]:
def moving_avg_np(x, w): # numpy reference
    c = np.cumsum(x)
    return (c[w-1:] - np.concatenate([np.zeros(1, x.dtype), c[:-w]])) / w

def moving_avg(x, w):
    # TODO: c = xp.cumsum(x); (c[w:]-c[:-w])/w  
    # 앞부분 처리 포함 : (c[w-1:] - xp.concatenate([xp.zeros(1, x.dtype), c[:-w]])) / w
    raise NotImplementedError

x_np = np.arange(20, dtype=np.float32)
# print(cp.asnumpy(moving_avg(cp.asarray(x_np), 5)))
# compare('moving_avg', lambda: moving_avg_np(x_np, 5), lambda: moving_avg(cp.asarray(x_np), 5), n_repeat=5, n_warmup=2)

<details><summary>💡 해답 보기</summary>

```python
def moving_avg(x, w):
    xp = cp.get_array_module(x)
    c = xp.cumsum(x)
    return (c[w-1:] - xp.concatenate([xp.zeros(1, x.dtype), c[:-w]])) / w

x_np = np.arange(20, dtype=np.float32)
print(cp.asnumpy(moving_avg(cp.asarray(x_np), 5)))
```
</details>

포인트: `c[w-1:]`는 구간의 끝점 누적합, `c[:-w]`는 그보다 `w`칸 앞선 누적합입니다(맨 앞은 0으로 패딩).
두 배열을 빼면 정확히 길이 `w`짜리 구간합이 남고, `w`로 나누면 이동평균이 됩니다. 슬라이싱과 뺄셈만으로
끝나는 벡터화된 구현이라 파이썬 반복문 없이 GPU에서 그대로 병렬 처리됩니다.

<a id="2"></a>
## 2. 선형대수 `cupy.linalg` + PCA

📖 [`cupy.linalg`](https://docs.cupy.dev/en/stable/reference/linalg.html) — `solve`, `svd`, `eigh`, `qr`, `inv`, `norm`, `lstsq` 등 (cuSOLVER/cuBLAS)

행렬곱·노름처럼 비교적 단순한 연산은 **cuBLAS**가, `solve`/`svd`/`eigh`/`qr`처럼 분해(decomposition)가
필요한 연산은 **cuSOLVER**가 담당합니다. 두 라이브러리 모두 NVIDIA가 자사 GPU 아키텍처에 맞춰 직접
튜닝한 것이라, CuPy는 얇은 파이썬 래퍼로 그 성능을 그대로 물려받습니다.

### 🔬 이론 배경 — 선형대수 & PCA

1500차원의 거대한 무작위 행렬을 만들어서,
1) CPU와 GPU가 똑같이 연립방정식을 잘 푸나 테스트(solve)해보고,
2) GPU로 행렬의 특이값(Singular Values)을 뽑아내어
3) 이 행렬이 얼마나 안정적인 구조인지 확인(cond)하는 벤치마크 및 검증 코드 예제입니다.

- **solve(Ax=b)**: A를 LU로 분해해 삼각계 후진대입으로 해를 구함(대략 O(n³)).
- **SVD** `A = U Σ Vᵀ`: 특이값 Σ가 데이터의 '주축 크기' → 차원축소·저랭크 근사.
- **eigh**(대칭 고유분해)로 **PCA**: 공분산행렬의 **고유벡터 = 분산이 최대인 방향**, 고유값 = 그 방향의 분산량.

### 조금 더 구체적으로

**계산량 감(感) 잡기**: `n=1500`인 `solve`는 대략 `(2/3)n³ ≈ 2.25×10⁹` 부동소수점 연산(FLOP)이 필요합니다.
현대 GPU의 float32 연산 성능이 수~수십 TFLOPS(초당 10¹²회 연산)임을 감안하면 이론상 1ms 안팎이면 끝날
연산이지만, 실제로는 데이터 전송·커널 launch·LU 분해의 순차 의존성(피벗팅 등) 때문에 이론치보다 항상
느립니다 — "이론 성능과 실측 성능의 괴리"는 `01_benchmark_basics`·`05_memory_profiling`에서 반복해서
확인하게 될 주제입니다.

**조건수(condition number, `cond(A) = σ_max/σ_min`)**: SVD의 최대/최소 특이값 비율로, 이 행렬로 정의된
선형계가 입력의 작은 오차를 얼마나 증폭시키는지를 나타냅니다. `cond(A)`가 크면(예: 10⁶ 이상) `solve`의
결과가 반올림 오차에 민감해져 CPU/GPU 결과 차이도 커질 수 있습니다 — 위에서 다룬 "CPU/GPU 결과가 다른
이유"가 조건수가 큰 행렬일수록 더 두드러지는 이유이기도 합니다. 이 실습에서는 무작위 정규분포 행렬을
쓰므로 `cond(A)`가 보통 수백~수천 수준으로 나오는 것이 정상입니다.

**PCA와 SVD의 관계**: 사실 PCA는 공분산행렬을 `eigh`로 고유분해하는 방법 외에, 중심화된 데이터 행렬
`Xc`를 직접 SVD(`Xc = U Σ Vᵀ`)해서 구할 수도 있습니다(`V`의 열이 주성분, `Σ²/(N-1)`이 고유값). 후자가
수치적으로 더 안정적이라고 알려져 있지만, 이 노트북에서는 "공분산 행렬 → 고유분해"라는 더 널리 쓰이는
교과서적 정의를 그대로 구현합니다(아래 PCA 미니앱).

In [ ]:
# 기본기: solve / svd / eigh / norm
n = 1500
A_np = np.random.randn(n, n).astype(np.float32); b_np = np.random.randn(n).astype(np.float32)
A_cp, b_cp = cp.asarray(A_np), cp.asarray(b_np)
allclose(np.linalg.solve(A_np, b_np), cp.linalg.solve(A_cp, b_cp), rtol=2e-3, atol=2e-3, name='solve')
S = cp.linalg.svd(A_cp, compute_uv=False)
print('cond(A)~', float(S[0]/S[-1]), '| ||A||_2 =', float(S[0]))
compare('svd', lambda: np.linalg.svd(A_np, compute_uv=False), 
               lambda: cp.linalg.svd(A_cp, compute_uv=False), n_repeat=5, n_warmup=2)

**배치 행렬곱**: 스택된 `(B, n, n)` 행렬을 한 번에 곱하면(`cp.matmul`) GPU가 특히 강합니다.

바로 아래 실습에서는 `B=64`개의 `256×256` 행렬을 한 번에 곱합니다. 개별 곱 하나가 `2×256³ ≈ 3.4×10⁷` FLOP이니
64개를 합치면 총 `~2.1×10⁹` FLOP 규모입니다. 이걸 파이썬 `for b in range(64): C[b] = A[b] @ B[b]`처럼
**64번의 개별 커널 launch**로 처리하면 launch당 고정 오버헤드(수 마이크로초, `00`의 "일회성 오버헤드"
논의 참고)가 64번 누적되지만, `cp.matmul`에 3차원 배치를 통째로 넘기면 cuBLAS가 이를 **batched GEMM**
한 번의 호출로 처리해 오버헤드를 크게 줄입니다. "여러 개의 작은 작업을 하나로 묶어 GPU에 한 번에
넘긴다"는 이 패턴은 이후 스트림·커널 융합(`06_streams_async`, 이번 노트북 5절의 `@cupy.fuse`)에서
반복해서 등장하는 핵심 아이디어입니다.

In [6]:
Bm, n = 64, 256
A = cp.random.random((Bm, n, n), dtype=cp.float32)
Bb = cp.random.random((Bm, n, n), dtype=cp.float32)
Cb = cp.matmul(A, Bb)              # 배치 matmul
print('batched matmul out:', Cb.shape)
A_np = cp.asnumpy(A); B_np = cp.asnumpy(Bb)
compare('bmm', lambda: np.matmul(A_np, B_np), lambda: cp.matmul(A, Bb), n_repeat=5, n_warmup=2)

batched matmul out: (64, 256, 256)
             bmm | CPU     5.406 ms | GPU     0.038 ms | 142.35x


(5.405934900045395, 0.03797560930252075, 142.35281538159703)

### 미니앱 — PCA (주성분 분석)
공분산 행렬의 고유분해(`eigh`)로 주성분을 구합니다. (고유벡터는 부호 모호성이 있어 **고유값**으로 검증)

PCA(주성분 분석)는 고차원 데이터를 **분산이 가장 큰 방향**부터 순서대로 나열해, 정보 손실을 최소화하며
차원을 줄이는 대표적 기법입니다. 데이터 압축, 노이즈 제거, 시각화(2~3차원으로 투영), 이후 모델 학습의
전처리 단계 등에 광범위하게 쓰입니다. 알고리즘은 세 단계입니다: (1) 각 특징의 평균을 빼 **중심화**,
(2) 특징 간 공분산행렬 `C = XᵀX/(N-1)` 계산, (3) `C`를 고유분해해 고유값이 큰 순서대로 고유벡터(주성분)를
정렬. `D=20`차원 데이터에서 `k=5`개만 뽑으면 `(N, 20) → (N, 5)`로 차원이 75% 줄어드는 셈입니다.

> ⚠️ **고유벡터의 부호 모호성**: `Cv = λv`가 성립하면 `C(-v) = λ(-v)`도 성립하므로, 고유벡터는 부호가
> 뒤집힌 채로 반환되어도 수학적으로 틀리지 않습니다. CPU/GPU가 내부적으로 다른 하우스홀더 반사
> 순서를 거치면 같은 고유값에 대해 **부호가 반대인 고유벡터**를 돌려줄 수 있습니다. 그래서 이 실습은
> 부호 영향을 받지 않는 **고유값**만으로 CPU/GPU 결과 일치를 검증합니다.

In [ ]:
def pca_np(X, k): # numpy reference
    # -------------------------------------------------------------------------
    # 1. 중심화 (Centering)
    #    - 각 열(특징)의 평균을 구해서 X에서 빼줍니다.
    #    - X.mean(axis=0, keepdims=True)을 활용해 브로드캐스팅(차원 맞춤)
    #    - Shape: (N, D) -> (N, D)
    Xc = X - X.mean(axis=0, keepdims=True)

    # 2. 공분산 행렬 계산 (Covariance Matrix)
    #    - 공식: C = (Xc.T @ Xc) / (N - 1), N은 데이터의 개수(X.shape[0])입니다.
    #    - Shape: (D, N) @ (N, D) -> (D, D)
    Cov = (Xc.T @ Xc) / (X.shape[0] - 1)

    # 3. 고유분해 (Eigen Decomposition)
    #    - 함수: np.linalg.eigh(C)  (xp는 np 또는 cp)
    #    - 반환값: w(고유값 배열), V(고유벡터 행렬)
    #    - 주의: eigh는 '오름차순(작은 값->큰 값)'으로 정렬되어 나옵니다!
    w, V = np.linalg.eigh(Cov)            # 오름차순

    # 4. 상위 k개 고유값 및 주성분 추출 (Sorting & Slicing)
    #    - 내림차순(큰 값->작은 값)으로 뒤집은 뒤, 앞의 k개만 잘라내야 합니다.
    #    - 배열 뒤집기: xp.argsort(w)[::-1][:k]
    #    - comps (주성분) Shape: V에서 해당 인덱스의 열만 선택 -> (D, k)
    idx = np.argsort(w)[::-1][:k]
    comps = V[:, idx]

    # 5. 차원 축소 투영 (Projection)
    #    - 공식: Xc @ comps
    #    - Shape: (N, D) @ (D, k) -> (N, k)
    return Xc @ comps, w[idx]

def pca(X, k):
    # TODO: 중심화 -> 공분산 C=(Xc.T@Xc)/(N-1) -> eigh -> 상위 k 고유값/주성분 -> 투영(Xc@comps)
    #       반환: (투영 (N,k), 상위 k 고유값 내림차순)
    raise NotImplementedError

X_np = (np.random.randn(5000, 20) @ np.random.randn(20, 20)).astype(np.float32)
# _, val_np = pca(X_np, 5)
# _, val_cp = pca(cp.asarray(X_np), 5)
# allclose(val_np, cp.asnumpy(val_cp), rtol=1e-2, atol=1e-2, name='PCA eigenvalues')
# compare('PCA', lambda: pca_np(X_np, 5), lambda: pca(cp.asarray(X_np), 5), n_repeat=5, n_warmup=2)

<details><summary>💡 해답 보기</summary>

```python
def pca(X, k):
    xp = cp.get_array_module(X)
    Xc = X - X.mean(axis=0, keepdims=True)
    Cov = (Xc.T @ Xc) / (X.shape[0] - 1)
    w, V = xp.linalg.eigh(Cov)            # 오름차순
    idx = xp.argsort(w)[::-1][:k]
    comps = V[:, idx]
    return Xc @ comps, w[idx]
```
</details>

포인트: `eigh`는 대칭행렬 전용 고유분해로, 일반 `eig`보다 빠르고 수치적으로 안정적이며 고유값을
항상 **오름차순 실수**로 반환합니다. `argsort(w)[::-1][:k]`로 내림차순 상위 `k`개 인덱스를 뽑아
고유값과 고유벡터(주성분) 양쪽에 동일하게 적용하는 것이 핵심입니다. 검증 시 `rtol=atol=1e-2`로 다른
셀보다 허용오차를 넉넉히 잡은 이유는, 고유분해가 공분산행렬(리덕션의 결과)에 대한 반복적 수치 알고리즘이라
CPU/GPU 연산 순서 차이가 누적되기 쉽기 때문입니다(위 이론 배경 셀의 오차 표 참고).

**연습 — 최소제곱(lstsq)**: 과결정계 `Ax≈b`를 `linalg.lstsq`로 푸세요.

1. 문제 정의
- `A`: (2000, 50) 행렬 → **방정식 2000개, 미지수 50개**
- `b`: (2000,) 벡터

방정식 개수(2000)가 미지수 개수(50)보다 훨씬 많은 **과결정계(overdetermined system)**입니다. 이런 경우 보통 `Ax = b`를 정확히 만족하는 해가 존재하지 않기 때문에, 오차를 최소화하는 근사해를 구해야 합니다.

2. 최소제곱해 (Least Squares Solution)
`xp.linalg.lstsq(A, b, rcond=None)[0]`는 다음을 최소화하는 `x`를 찾습니다.

$$
\min_x \; \| Ax - b \|_2^2
$$

내부적으로 SVD(특이값 분해)나 QR 분해를 이용해 계산하며, `rcond=None`은 특이값 중 무시할 정도로 작은 값(rank 판정 기준)을 라이브러리 기본값(머신 epsilon 기반)으로 설정하는 옵션입니다 (경고 메시지 방지 목적).

### 조금 더 구체적으로 — `solve`와 뭐가 다른가

`solve`는 `A`가 **정방행렬이고 정확한 해가 존재한다고 가정**하는 반면, `lstsq`는 방정식 수와 미지수
개수가 달라 정확한 해가 없을 수도 있는 상황을 다룹니다. 내부적으로 SVD를 쓰는 경우 `A = UΣVᵀ`로
분해한 뒤 `x = V Σ⁺ Uᵀ b` (Σ⁺는 특이값의 역수로 이뤄진 유사역행렬)로 계산합니다. `rcond`는 이 Σ⁺를
구할 때 "0으로 나누는" 사고를 막기 위해 지나치게 작은 특이값을 0 취급하는 임계값이며, 지정하지 않으면
(향후 NumPy 버전에서 기본 동작이 바뀌는 것을 막기 위해) 경고가 뜰 수 있어 명시적으로 `None`을 넘기는
것이 관례입니다.


In [ ]:
def solve_lstsq_np(A, b): # numpy reference
    return np.linalg.lstsq(A, b, rcond=None)[0]

def solve_lstsq(A, b):
    # TODO: xp.linalg.lstsq(A, b, rcond=None)[0]
    raise NotImplementedError
# A=np.random.randn(2000,50).astype('f4'); b=np.random.randn(2000).astype('f4')
# allclose(solve_lstsq(A,b), cp.asnumpy(solve_lstsq(cp.asarray(A),cp.asarray(b))), rtol=2e-2, atol=2e-2, name='lstsq')
# compare('lstsq', lambda: solve_lstsq_np(A, b), lambda: solve_lstsq(cp.asarray(A), cp.asarray(b)), 
#           n_repeat=5, n_warmup=2)

<details><summary>💡 해답 보기</summary>

```python
def solve_lstsq(A, b):
    xp = cp.get_array_module(A)
    return xp.linalg.lstsq(A, b, rcond=None)[0]
```
</details>

포인트: `lstsq`가 반환하는 튜플은 `(해, 잔차제곱합, rank, 특이값)` 순서이며, 여기서는 첫 번째 원소인
**해(x)**만 사용합니다(`[0]`). `rcond=None`을 명시해 rank 판정 임계값을 라이브러리 기본값으로 고정하는
것도 함수 시그니처를 장치 비종속으로 유지하는 데 도움이 됩니다.

<a id="3"></a>
## 3. FFT `cupy.fft` + 2D 디노이즈

📖 [`cupy.fft`](https://docs.cupy.dev/en/stable/reference/fft.html) — `fft/ifft`, `rfft/irfft`, `fft2/ifft2`, `fftfreq`, `fftshift`

`cupy.fft`는 NVIDIA **cuFFT**를 그대로 호출합니다. cuFFT는 내부적으로 계획(plan)을 캐싱해 같은
shape/dtype의 변환을 반복 호출할 때 재사용하므로, 반복되는 FFT 연산에서는 첫 호출 이후 속도가
안정화됩니다(`00`의 "일회성 오버헤드·워밍업" 논의와 같은 맥락). `04_scipy_routines`에서는 같은
cuFFT 백엔드 위에서 동작하는 `cupyx.scipy.fft`로 DCT 기반 압축을 다룹니다 — 이번 절의 `fft2`/`fftshift`
조합에 익숙해두면 그쪽 이해가 훨씬 수월합니다.

### 🔬 이론 배경 — FFT가 하는 일
- **DFT**는 신호를 주파수 성분으로 분해: `X_k = Σ_n x_n · e^(−2πi·kn/N)`.
- 직접 계산은 O(N²), **FFT**는 분할정복으로 **O(N log N)**.
- `rfft`는 실수 입력의 대칭성을 이용해 계산·저장을 절반으로.
- **컨볼루션 정리**: 시간영역 컨볼루션 = 주파수영역 **곱** → 큰 필터를 FFT로 빠르게 적용.

### 조금 더 구체적으로 — 숫자로 보는 FFT의 이득

바로 아래 실습은 길이 50만(`5×10⁵`) 신호와 길이 2048짜리 필터를 컨볼루션합니다. 직접 컨볼루션(시간영역에서
슬라이딩 곱셈-합)은 대략 `O(N·M) ≈ 5×10⁵ × 2048 ≈ 10⁹`번의 곱셈-덧셈이 필요합니다. 반면 FFT 기반
컨볼루션은 두 신호를 합친 길이 `N' ≈ N+M`에 대해 FFT 한 번(`O(N' log N')`) + 주파수영역 원소별 곱
(`O(N')`) + 역FFT 한 번(`O(N' log N')`)이면 끝나, `N'log₂N' ≈ 5×10⁵ × 19 ≈ 10⁷` 수준으로 **약 100배
적은 연산**입니다. 필터가 길어질수록(`M`이 클수록) 이 격차는 더 벌어집니다 — "필터가 크면 FFT 컨볼루션이,
필터가 아주 작으면(예: 3×3 커널) 직접 컨볼루션이 유리하다"는 것이 일반적인 경험칙입니다.

`rfft`가 저장을 절반으로 줄이는 이유는 **에르미트 대칭성**(실수 입력의 DFT는 `X_k = conj(X_{N-k})`를
만족) 때문입니다. 그래서 뒤쪽 절반은 앞쪽 절반의 켤레복소수로 완전히 복원 가능해, `rfft`는 길이 `N`
실수 입력에 대해 길이 `N//2+1`개의 복소수만 계산·저장합니다 — 메모리와 연산량 모두 거의 절반으로 줄어드는 셈입니다.

**2D 디노이즈에서 쓰는 `fftshift`/`ifftshift`**: `fft2`의 결과는 기본적으로 "0 주파수가 배열의 (0,0)
모서리"에 오도록 배치됩니다. `fftshift`는 이를 배열 **중앙**으로 옮겨 사람이 보기 편한 형태로
재배치하는 함수이고(`ifftshift`는 그 역), 아래 미니앱에서 "중앙 사각형만 남기고 마스킹"하는 작업을
직관적으로 만들어줍니다.

In [ ]:
# 1D: rfft/irfft 로 FFT 컨볼루션 (장치 비종속)
def fftconv(x, h):
    xp = cp.get_array_module(x); n = x.size + h.size - 1
    return xp.fft.irfft(xp.fft.rfft(x, n) * xp.fft.rfft(h, n), n)
x_np = np.random.randn(500_000).astype(np.float32); h_np = np.exp(-np.linspace(0,8,2048)).astype(np.float32)
allclose(fftconv(x_np,h_np), fftconv(cp.asarray(x_np),cp.asarray(h_np)), rtol=2e-3, atol=2e-3, name='fftconv')
compare('fftconv', lambda: fftconv(x_np,h_np), lambda: fftconv(cp.asarray(x_np),cp.asarray(h_np)), 
        n_repeat=5, n_warmup=2)

### 미니앱 — 2D 주파수 저역통과 디노이즈
이미지를 `fft2`→`fftshift` 후 중앙(저주파) 사각형만 남기고 역변환하면 고주파 잡음이 제거됩니다.

작동 원리:

1. 이미지를 주파수 세상으로 보냅니다 (FFT 변환).
2. 주파수 세상의 가운데(주파수가 0인 부근)에는 '부드러운 성분(저주파)'이 모여 있고, 바깥쪽에는 '거칠고 자글자글한 성분(고주파 잡음)'이 모여 있습니다.
3. 가운데 중심에서 'radius(반지름)' 크기만큼의 사각형만 남기고 바깥쪽은 0으로 다 지웁니다 (마스크 적용).
4. 깨끗해진 주파수를 다시 원래 이미지 세상으로 되돌립니다 (IFFT 역변환).

### 조금 더 구체적으로 — `radius`가 의미하는 것

`radius(r)`가 작을수록 중앙에 남기는 저주파 성분이 적어져 **더 많이 부드러워지지만**(디테일 손실도 커짐),
너무 작으면 원래 신호(`sin(xx/8)` 패턴)의 성분까지 잘려나가 이미지가 뭉개집니다. 반대로 `r`이 너무 크면
고주파 잡음이 충분히 제거되지 않습니다. 128×128 이미지에서 `r=12`는 전체 주파수 대역의 약 `24/128 ≈ 19%`
폭만 남기는 셈으로, "신호는 살리고 잡음은 죽이는" 균형점을 실습으로 직접 관찰하는 것이 이 미니앱의
포인트입니다. 이렇게 **사각형(box) 마스크**로 딱 자르면 주파수 경계에서 불연속이 생겨 복원된 이미지에
약한 물결무늬(**링잉, ringing / Gibbs 현상**)가 남을 수 있는데, 실전에서는 가우시안처럼 부드럽게
감쇠하는 마스크를 써서 이를 줄이기도 합니다. `04_scipy_routines`의 `ndimage` 절에서 공간영역 필터
(가우시안 블러 등)와 비교해보면 "같은 저역통과라도 주파수영역 vs 공간영역 구현"의 차이를 체감할 수 있습니다.

In [ ]:
def lowpass2d_np(img, r): # numpy reference

    # Step 1: 이미지를 주파수 공간으로 변환하고, 중심이 가운데로 오도록 셔플(shift)합니다.
    # -> Shape 변환 없음: (H, W) 복소수 행렬
    F = np.fft.fftshift(np.fft.fft2(img))

    # Step 2: 검은색 사각형 필터(초기값 0)를 만듭니다.
    h, w = img.shape
    m = np.zeros((h, w), dtype=img.dtype)

    # Step 3: 정가운데 1/2 영역에만 흰색 네모(1)를 그립니다. (이 안의 주파수만 살아남음)
    cy, cx = h//2, w//2
    m[cy-r:cy+r, cx-r:cx+r] = 1

    # Step 4: 주파수 정보와 마스크를 곱해 고주파수 영역의 잡음을 날려버립니다.
    # Step 5: 역변환을 통해 다시 눈에 보이는 이미지로 되돌린 후, 실수(Real) 값만 취합니다.
    return np.real(np.fft.ifft2(np.fft.ifftshift(F * m)))

def lowpass2d(img, r): 
    # TODO: F=fftshift(fft2(img)); 중앙 (2r x 2r)만 남기고 0; real(ifft2(ifftshift(F)))
    raise NotImplementedError

rng = np.random.default_rng(0)
yy, xx = np.mgrid[0:128, 0:128]
img_np = (np.sin(xx/8.0) + 0.6*rng.standard_normal((128,128))).astype(np.float32)
# out_np = lowpass2d(img_np, 12); out_cp = cp.asnumpy(lowpass2d(cp.asarray(img_np), 12))
# allclose(out_np, out_cp, rtol=2e-3, atol=2e-3, name='lowpass2d')
# compare('lowpass2d', lambda: lowpass2d_np(img_np, 12), 
#                      lambda: lowpass2d(cp.asarray(img_np), 12), n_repeat=5, n_warmup=2)

<details><summary>💡 해답 보기</summary>

```python
def lowpass2d(img, r):
    xp = cp.get_array_module(img)
    F = xp.fft.fftshift(xp.fft.fft2(img))
    h, w = img.shape; cy, cx = h//2, w//2
    m = xp.zeros((h, w), dtype=img.dtype)
    m[cy-r:cy+r, cx-r:cx+r] = 1
    return xp.real(xp.fft.ifft2(xp.fft.ifftshift(F * m)))
```
</details>

포인트: 마스크 `m`은 실수(float) 배열이지만 `F`는 복소수(complex) 배열입니다 — `F * m`처럼 복소수와
실수를 곱하면 NumPy/CuPy가 자동으로 타입을 승격(promote)해 복소수 결과를 만들어줍니다. `ifftshift`는
`fftshift`의 정확한 역연산이므로, 마스킹 후 반드시 `ifftshift`로 되돌린 다음 `ifft2`를 호출해야
주파수 배치가 올바르게 복원됩니다. 순서를 잊고 `ifftshift` 없이 바로 `ifft2`를 호출하면 이미지가
좌우/상하로 뒤바뀐 채 복원되니 주의하세요.

In [ ]:
# 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_np, cmap='gray')
plt.title("1. Before (Original + Noise)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(out_np, cmap='gray')
plt.title("2. After (Lowpass, r=12)")
plt.axis('off')

plt.tight_layout()
plt.show()

<a id="4"></a>
## 4. 난수 `cupy.random` + 몬테카를로

📖 [`cupy.random`](https://docs.cupy.dev/en/stable/reference/random.html) — 권장 **Generator API** `cp.random.default_rng()` (cuRAND)

`cupy.random`은 NVIDIA **cuRAND**를 백엔드로 씁니다. NumPy와 마찬가지로 오래된 전역 상태 기반 함수
(`cp.random.seed()`, `cp.random.rand()` 등, legacy API)와 명시적으로 생성기 객체를 만들어 쓰는
**Generator API**(`cp.random.default_rng(seed)`)가 공존하는데, 이 노트북은 최신 권장 방식인 Generator
API를 사용합니다 — 전역 상태를 공유하지 않아 여러 시뮬레이션을 독립적으로 재현·병렬화하기 쉽습니다.

### 🔬 이론 배경 — 난수 & 몬테카를로
- **PRNG**: 시드로 결정되는 의사난수열(재현 가능, 참난수 아님).
- **몬테카를로**: 난수 표본으로 적분·확률을 추정. 표준오차 ~ **1/√N**(표본 수 N).
- 정확도를 높이려면 표본이 많이 필요 → **대량 난수 생성에 GPU가 유리**.

### 조금 더 구체적으로 — 왜 1/√N이고, 얼마나 많은 표본이 필요한가

몬테카를로 추정치의 표준오차가 `1/√N`으로 줄어드는 것은 **중심극한정리**의 직접적인 결과입니다: N개의
독립 표본 평균의 분산은 표본 하나의 분산을 N으로 나눈 값이므로, 표준편차(오차의 크기)는 `σ/√N`이 됩니다.
이 말은 오차를 **10분의 1로 줄이려면 표본을 100배** 늘려야 한다는 뜻이기도 합니다 — 아래 π 추정 미니앱에서
`N`을 `10⁵ → 10⁸`(1000배)로 늘렸을 때 오차가 대략 `√1000 ≈ 32`배만 줄어드는 것을 직접 관찰하게 됩니다.
정밀도를 한 자릿수 더 얻으려면 표본을 100배 더 뽑아야 하는 이 "수렴이 느린" 특성 때문에, 몬테카를로는
**표본 생성 자체의 처리량(throughput)**이 곧 정확도의 상한을 좌우합니다 — 초당 수억~수십억 개의 난수를
뽑아낼 수 있는 GPU(cuRAND)가 CPU 대비 크게 유리한 이유입니다. 반대로 표본 수가 적은 경우(예: N<10⁴)에는
난수 생성량 자체가 작아 GPU의 커널 launch 오버헤드가 이득을 상쇄할 수 있다는 점도 `01_benchmark_basics`의
손익분기 논의와 같은 맥락에서 기억해두면 좋습니다.

In [ ]:
rng = cp.random.default_rng(0)
print('normal :', cp.asnumpy(rng.standard_normal(3, dtype=cp.float32)))
print('int    :', cp.asnumpy(rng.integers(0, 10, size=5)))
print('choice :', cp.asnumpy(cp.random.choice(cp.arange(100), size=5, replace=False))) 
# 재현성: 같은 시드 -> 같은 수열 (같은 버전/디바이스)
a = cp.random.default_rng(1).standard_normal(4, dtype=cp.float32)
b = cp.random.default_rng(1).standard_normal(4, dtype=cp.float32)
print('재현?', bool((a==b).all()))

### 미니앱 — 몬테카를로로 π 추정
대량 난수는 GPU의 강점입니다. 단위정사각형에 점을 뿌려 1/4원 내부 비율로 π를 추정합니다.

**원리**: 한 변의 길이가 1인 정사각형 `[0,1]×[0,1]` 안에 균등하게 난수 점 `(x, y)`를 뿌리면, 원점을
중심으로 반지름 1인 사분원(넓이 `π/4`) 내부에 들어갈 확률은 정사각형 전체 넓이(1) 대비 `π/4`입니다.
즉 `x²+y² ≤ 1`을 만족하는 점의 비율을 `p`라 하면 `p ≈ π/4`이므로 `π ≈ 4p`로 추정할 수 있습니다.
아래 실습은 `N`을 `10⁵`에서 `10⁸`까지 늘려가며 추정치가 실제 `π=3.14159...`에 얼마나 빨리 수렴하는지
(그리고 그 수렴 속도가 정확히 `1/√N` 패턴을 따르는지) 직접 확인합니다.

In [ ]:
def mc_pi_np(n, seed=0): # numpy reference
    rng = np.random.default_rng(seed)
    x = rng.random(n); y = rng.random(n)
    inside = (x*x + y*y <= 1).sum()
    return 4 * inside / n

def mc_pi(n, seed=0):
    # TODO: rng=cp.random.default_rng(seed); x,y 난수; inside=(x*x+y*y<=1).sum(); 4*inside/n
    raise NotImplementedError

# for n in [10**5, 10**6, 10**7, 10**8]:
#     est = mc_pi(n); print(f'N={n:>10,}  π~{est:.5f}  err={abs(est-math.pi):.2e}')
#     compare('mc_pi', lambda: mc_pi_np(10**7, seed=0), 
#                      lambda: mc_pi(10**7, seed=0), n_repeat=2, n_warmup=1)

<details><summary>💡 해답 보기</summary>

```python
def mc_pi(n, seed=0):
    rng = cp.random.default_rng(seed)
    x = rng.random(n, dtype=cp.float32); y = rng.random(n, dtype=cp.float32)
    inside = ((x*x + y*y) <= 1.0).sum()
    return float(4 * inside / n)
# 오차가 ~1/sqrt(N) 로 줄어드는지 관찰하세요.
```
</details>

포인트: `(x*x + y*y) <= 1.0`은 각 점이 사분원 안에 있는지를 나타내는 불리언 배열이고, `.sum()`은 그
불리언(0/1)을 리덕션으로 합산해 내부 점의 개수를 셉니다. `N=10⁸`(1억)개의 난수를 뽑고 판정하는 이 작업
전체가 파이썬 반복문 없이 GPU 커널 몇 개로 끝난다는 점이 핵심이며, 표 형태로 출력되는 오차가
`N`이 100배 늘 때마다 약 10배(=√100)씩 줄어드는지 직접 확인해보세요 — 이론 배경 셀에서 다룬 `1/√N`
수렴 법칙의 실증입니다.

<a id="5"></a>
## 5. 커널 퓨전 `@cupy.fuse`

📖 [`cupy.fuse`](https://docs.cupy.dev/en/stable/reference/generated/cupy.fuse.html) — 여러 원소/리덕션 연산을 **하나의 커널로 융합**해 중간배열(temporary)과 커널 런치를 줄입니다.
처음 호출 시 dtype/ndim에 맞춰 커널을 컴파일·캐시하므로, **같은 데코레이트 함수를 재사용**하세요. (Day 2 커널 작성의 예고편)

1. a*a, b*b, +. log1p 연산에 대해서 4개의 커널을 순차적으로 실행, 커널 수행 오버헤드가 큼

```python
def elementwise_plain(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)
```

2. a*a, b*b, +. log1p 연산에 대해서 4개의 커널을 한번에 모아서 실행

```python
@cp.fuse()
def elementwise_fused(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)
```

### 조금 더 구체적으로 — temporary가 얼마나 사라지는가

`elementwise_plain`은 `a*a`, `b*b`, `a*a+b*b`, `sqrt(...)`, `log1p(c)`, 마지막 덧셈까지 순서대로
**최소 5~6개의 중간(temporary) 배열**을 만들고, 그때마다 별도의 CUDA 커널을 launch합니다. 원소 3천만
(`30_000_000`)개짜리 float32 배열 하나의 크기는 `30,000,000 × 4바이트 ≈ 114MB`이므로, 중간 배열이
여러 개 생기면 GPU 메모리에 수백 MB를 쓰고 지우는 트래픽이 추가로 발생합니다. 이런 연산은 계산량 자체는
크지 않고 **메모리 대역폭이 병목**(memory-bound)인 경우가 많아서, 중간 배열을 오가는 읽기/쓰기 트래픽이
곧 실행 시간의 대부분을 차지합니다.

`@cp.fuse()`는 데코레이트한 함수 전체를 분석해 **하나의 커널**로 합칩니다. 그 결과 (1) 커널 launch
횟수가 4~6번에서 1번으로 줄고(`00`에서 다룬 launch 오버헤드가 그만큼 줄어듦), (2) 중간 결과를 GPU
메모리에 왕복시키지 않고 **레지스터/캐시 안에서 바로 다음 연산에 사용**하므로 메모리 트래픽 자체가
크게 줄어듭니다. 이것이 바로 아래 벤치마크에서 fused 버전이 더 빠르게 나오는 이유입니다. 이 "여러 연산을
하나의 커널로 합쳐 메모리 왕복을 줄인다"는 아이디어는 Day 2에서 직접 커널을 작성할 때
(`07_cupy_kernels`의 `ElementwiseKernel`/`RawKernel`) 훨씬 세밀하게 제어할 수 있게 됩니다 — `@cp.fuse`는
그 개념을 데코레이터 한 줄로 자동화해주는 진입점인 셈입니다.

> ⚠️ `@cp.fuse()`는 원소별(elementwise) 연산과 일부 리덕션의 조합만 융합할 수 있습니다. 조건 분기나
> 형상을 바꾸는 연산(`reshape`, `sort` 등)이 섞이면 융합되지 않거나 예상과 다르게 동작할 수 있으니,
> 순수한 수식 형태의 함수에 우선 적용해보는 것이 안전합니다.

In [ ]:
def elementwise_plain(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)

@cp.fuse()
def elementwise_fused(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)

a = cp.random.random(30_000_000, dtype=cp.float32)
b = cp.random.random(30_000_000, dtype=cp.float32)
c = cp.random.random(30_000_000, dtype=cp.float32)
_ = elementwise_fused(a, b, c)   # 워밍업(첫 호출에 커널 컴파일)
r1 = bench(lambda: elementwise_plain(a,b,c), n_repeat=20, name='plain')
r2 = bench(lambda: elementwise_fused(a,b,c), n_repeat=20, name='fused')
print(f'plain {gpu_ms(r1):.3f} ms | fused {gpu_ms(r2):.3f} ms | speedup {gpu_ms(r1)/gpu_ms(r2):.2f}x')

**연습 — 함수 융합**: 아래 `f`를 `@cp.fuse()`로 융합하고 정확성·속도를 확인하세요.

`tanh`·`exp`는 사칙연산보다 계산 비용이 큰 초월함수(transcendental function)라 원소당 계산량 자체도
어느 정도 있지만, 위에서 본 것처럼 중간 배열 왕복 비용은 여전히 별개로 존재합니다. `f_fused`를 정의해
`allclose`로 `f`와 결과가 (부동소수점 오차 범위 내에서) 같은지 확인하고, `bench`로 두 버전의 속도를
비교해보세요.

In [ ]:
def f(x, y):
    return (cp.tanh(x) + 1.0) * cp.exp(-y*y)

# TODO: @cp.fuse() 로 f_fused 정의 후 동일 결과·속도 비교

#x = cp.random.random(30_000_000, dtype=cp.float32)
#y = cp.random.random(30_000_000, dtype=cp.float32)
#allclose(f(x,y), f_fused(x,y), rtol=1e-5, atol=1e-5, name='fuse')
#_ = f_fused(x,y)
#print('plain', gpu_ms(bench(lambda: f(x,y))), 'ms | fused', gpu_ms(bench(lambda: f_fused(x,y))), 'ms')
#compare('fuse', lambda: f(x,y), lambda: f_fused(x,y), n_repeat=5, n_warmup=1)

<details><summary>💡 해답 보기</summary>

```python
def f(x, y):
    return (cp.tanh(x) + 1.0) * cp.exp(-y*y)

@cp.fuse()
def f_fused(x, y):
    return (cp.tanh(x) + 1.0) * cp.exp(-y*y)
```
</details>

포인트: `f`와 `f_fused`는 **완전히 같은 수식**을 담고 있고, 차이는 오직 `@cp.fuse()` 데코레이터
유무입니다 — 즉 사용자 입장에서는 함수 본문을 바꾸지 않고 데코레이터 한 줄만 추가해 커널 융합의
이득을 얻는 셈입니다. `_ = f_fused(x,y)`로 한 번 워밍업 호출을 해두는 이유는 `@cp.fuse()`가 **첫 호출
시점에** 실제 입력의 dtype/shape을 보고 융합 커널을 컴파일하기 때문입니다(이 노트북 5절 도입부 및
`00`의 "일회성 오버헤드" 논의 참고) — 이 컴파일 비용이 벤치마크에 섞여 들어가지 않도록 미리 소모해두는 것입니다.

<a id="6"></a>
## 6. 체크포인트

- [ ] `cupy.*` 모듈 함수(집계/정렬/탐색/누적/집합)를 사용했다
- [ ] `cupy.linalg`로 solve/svd/eigh와 **PCA**를 구현했다
- [ ] 배치 `matmul`로 GPU 강점을 확인했다
- [ ] `cupy.fft`로 1D 컨볼루션과 **2D 디노이즈**를 했다
- [ ] `cupy.random`으로 **몬테카를로 π**를 추정했다(오차~1/√N)
- [ ] `@cupy.fuse`로 원소 연산을 융합해 속도를 높였다

다음: **`04_scipy_routines`** — SciPy 루틴(fft·linalg·ndimage·sparse·signal·…).